# Mergen-SNN: Google Drive Tabanlı Kesintisiz Eğitim Notebook'u

Bu notebook, **Mergen-SNN** (~9M-20M Stage 4 Small SNN) dil modelinin Google Drive'a yüklenmiş proje üzerinden doğrudan ve kesintisiz eğitilmesi için tasarlanmıştır.

### Temel Özellikler:
- **git clone Yok:** Proje doğrudan Drive üzerindeki dizinden (`MyDrive/mergen-snn` veya `Drive/mergen-snn`) okunur.
- **Lokal Disk Hızı:** Hızlı G/Ç performansı için proje `/content/mergen-snn` lokal NVMe diskine aktarılır (1.25 GB ham veri atlanır, 69 MB binary hazır veri alınır).
- **Arka Plan Drive Senkronizasyonu:** Checkpoint'ler lokale atomik yazılır ve arka plandaki thread ile her 5 dakikada bir otomatik Drive'a kopyalanır.
- **Otomatik Resume:** Oturum kapansa dahi Drive'daki son checkpoint'ten otomatik devam edilir.

## Hücre 1: Google Drive Mount ve Proje Yolu Tespiti
Google Drive'ı bağlar ve projenin Drive üzerindeki konumunu otomatik olarak bulur.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Olası Drive yollarını dene
candidates = [
    '/content/drive/MyDrive/mergen-snn',
    '/content/drive/Drive/mergen-snn',
    '/content/drive/mergen-snn',
]
PROJECT_DIR = None
for p in candidates:
    if os.path.exists(p):
        PROJECT_DIR = p
        break
if PROJECT_DIR is None:
    raise RuntimeError("Proje klasörü Drive'da bulunamadı. Drive'da 'mergen-snn' klasörü olduğundan emin ol.")
print(f"Proje yolu: {PROJECT_DIR}")

## Hücre 2: Projeyi Lokale Kopyala (Hız İçin)
Hızlı disk G/Ç performansı için projeyi `/content/mergen-snn` dizinine kopyalar (büyük ham veriler hariç tutulur).

In [ ]:
import shutil, os
LOCAL_DIR = '/content/mergen-snn'
if os.path.exists(LOCAL_DIR):
    shutil.rmtree(LOCAL_DIR)

# Sadece gerekli klasörleri kopyala (data/raw hariç, çok büyük)
shutil.copytree(PROJECT_DIR, LOCAL_DIR, ignore=shutil.ignore_patterns(
    'raw', 'data/raw', 'checkpoints', '.venv', '__pycache__', '*.pyc', '.git', 'node_modules'
))
print("Proje lokale kopyalandı.")

## Hücre 3: Bağımlılıkları Kur ve Cython Uzantılarını Derle
`uv` paket yöneticisini kurar, kütüphaneleri senkronize eder ve C/C++ hızlandırma modüllerini derler.

In [ ]:
!pip install uv
!cd /content/mergen-snn && uv sync
!cd /content/mergen-snn && uv run python src/mergen/cython_ext/setup.py build_ext --inplace

## Hücre 4: Checkpoint Dizinlerini Hazırla ve Mevcut Modelleri Çek
Lokal ve Drive checkpoint dizinlerini oluşturur. Daha önce kaydedilmiş checkpoint varsa resume için lokale çeker.

In [ ]:
import glob, shutil, os
LOCAL_CKPT = '/content/mergen-snn/checkpoints'
DRIVE_CKPT = f'{PROJECT_DIR}/checkpoints'
os.makedirs(LOCAL_CKPT, exist_ok=True)
os.makedirs(DRIVE_CKPT, exist_ok=True)

# Drive'da önceden kaydedilmiş checkpoint varsa resume için lokale aktar
existing_ckpts = glob.glob(f'{DRIVE_CKPT}/*.pt')
if existing_ckpts:
    for f in existing_ckpts:
        shutil.copy2(f, LOCAL_CKPT)
    print(f"Drive'daki {len(existing_ckpts)} checkpoint resume için lokale aktarıldı.")
else:
    print("Drive'da henüz kayıtlı checkpoint yok, eğitim sıfırdan başlayacak.")

print(f"Lokal checkpoint: {LOCAL_CKPT}")
print(f"Drive checkpoint (yedek): {DRIVE_CKPT}")

## Hücre 5: Drive Senkronizasyon Thread'i (Arka Planda)
Her 5 dakikada bir lokalde oluşan yeni checkpoint'leri Google Drive'a yedekleyen arka plan iş parçacığıdır.

In [ ]:
import time, shutil, glob, os, threading
DRIVE_CKPT = f'{PROJECT_DIR}/checkpoints'
LOCAL_CKPT = '/content/mergen-snn/checkpoints'

def sync_loop():
    while True:
        try:
            files = glob.glob(f'{LOCAL_CKPT}/*.pt')
            for f in files:
                dest = os.path.join(DRIVE_CKPT, os.path.basename(f))
                if not os.path.exists(dest) or os.path.getmtime(f) > os.path.getmtime(dest):
                    shutil.copy2(f, dest)
                    print(f"[{time.strftime('%H:%M:%S')}] Drive'a kopyalandı: {os.path.basename(f)}")
        except Exception as e:
            print(f"Senkronizasyon hatası: {e}")
        time.sleep(300)  # 5 dakikada bir

# Arka planda başlat
threading.Thread(target=sync_loop, daemon=True).start()
print("Drive senkronizasyon thread'i başladı (her 5 dk).")

## Hücre 6: Eğitimi Başlat (Otomatik Resume ile)
Stage 4 Small SNN model eğitimini başlatır. Oturum kopsa bile `--resume auto` ile kaldığı adımdan devam eder.

In [ ]:
!cd /content/mergen-snn && uv run python scripts/train_small.py --resume auto

## Hücre 7: Eğitim Sonu Güvenli Senkronizasyon ve Doğrulama
Eğitim tamamlandığında veya durdurulduğunda tüm checkpoint'leri Drive'a tam senkronize eder.

In [ ]:
import glob, os, shutil

print("=== Eğitim Sonu Checkpoint Senkronizasyonu ===")
local_files = glob.glob('/content/mergen-snn/checkpoints/*.pt')
for f in local_files:
    dest = os.path.join(f'{PROJECT_DIR}/checkpoints', os.path.basename(f))
    shutil.copy2(f, dest)
    sz_mb = os.path.getsize(dest) / (1024 * 1024)
    print(f"  - Drive'da doğrulandı: {os.path.basename(dest)} ({sz_mb:.2f} MB)")

print(f"\nToplam {len(local_files)} checkpoint Google Drive üzerinde güvence altında!")